In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
path =r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.billing.csv'
df =spark.read.csv(path,header =True,inferSchema=True)
df.count()

In [0]:
# -----------------------------
# 1. Deduplication (Primary Key)
# -----------------------------
df_duplicate = df.dropDuplicates(["billing_id"])
df_duplicate.count()

In [0]:
# -----------------------------
# 2. Date Standardization
# -----------------------------
from pyspark.sql.functions import coalesce,try_to_date ,col
df_datestandard =( df_duplicate.withColumn(
        "bill_dt",
        coalesce(
            try_to_date(col("bill_date"), "dd-MM-yyyy"),
            try_to_date(col("bill_date"), "yyyy-MM-dd"),
            try_to_date(col("bill_date"), "MM-dd-yyyy"),
            try_to_date(col("bill_date"), "dd/MM/yyyy")
        )
    )
    .withColumn(
        "payment_dt",
        coalesce(
            try_to_date(col("payment_date"), "dd-MM-yyyy"),
            try_to_date(col("payment_date"), "yyyy-MM-dd"),
            try_to_date(col("payment_date"), "MM-dd-yyyy"),
            try_to_date(col("payment_date"), "dd/MM/yyyy")
        )
    ).drop("payment_date")               
    .drop("bill_date")
)

df_datestandard.display()


In [0]:
# -----------------------------
# 3. Range Validation Columns
# -----------------------------
from pyspark.sql.functions import when ,lit
from pyspark.sql.types import DoubleType
df_rangestandard = (df_datestandard.withColumn(
        "gross_amount_valid",
        when(col("gross_amount").cast(DoubleType()) > 0,
             col("gross_amount").cast(DoubleType()))
    )
    .withColumn(
        "discount_amount_valid",
        when(col("discount_amount").cast(DoubleType()) >= 0,
             col("discount_amount").cast(DoubleType()))
    )
    .withColumn(
        "ins_approved_valid",
        when(col("insurance_approved_amount").cast(DoubleType()) >= 0,
             col("insurance_approved_amount").cast(DoubleType()))
    )
    .withColumn(
        "oop_amount_valid",
        when(col("patient_out_of_pocket").cast(DoubleType()) >= 0,
             col("patient_out_of_pocket").cast(DoubleType()))
    )
    .withColumn(
        "outstanding_valid",
        when(col("outstanding_balance").cast(DoubleType()) >= 0,
             col("outstanding_balance").cast(DoubleType()))
    )
    .withColumn(
        "payment_received_valid",
        when(col("payment_received").cast(DoubleType()) >= 0,
             col("payment_received").cast(DoubleType()))
        .otherwise(lit(0.0))
    )
    .drop("gross_amount")
    .drop("discount_amount")
    .drop("insurance_approved_amount")
    .drop("patient_out_of_pocket")
    .drop("outstanding_balance") 
    .drop("payment_received")              
)

df_rangestandard.display()

In [0]:
# -----------------------------
# 4. Derived Columns
# -----------------------------
from pyspark.sql.functions import datediff
df_derived_columns =(df_rangestandard.withColumn(
        "net_payable",
        col("gross_amount_valid") - col("discount_amount_valid")
    )
    .withColumn(
        "patient_liability",
        col("net_payable") - col("ins_approved_valid")
    )
    .withColumn(
        "payment_lag_days",
        datediff(col("payment_dt"), col("bill_dt"))
    )
)

df_derived_columns.display()

In [0]:
# -----------------------------
# 5. Standardization Columns
# -----------------------------
from pyspark.sql.functions import initcap,col,trim,upper
df_standardized = (df_derived_columns.withColumn(
        "bill_status_std",
        when(col("bill_status").isNull(), "Unknown")
        .otherwise(initcap(trim(col("bill_status"))))
    )
    .withColumn(
        "payment_mode_std",
        when(col("payment_mode").isNull(), "UNKNOWN")
        .otherwise(upper(trim(col("payment_mode"))))
    )
    .drop("bill_status")
    .drop("payment_mode")
)

df_standardized.display()


In [0]:
# -----------------------------
# 6. Text Cleansing
# -----------------------------
df_textcleaning = (df_standardized.withColumn(
        "claim_number_clean",
        when(trim(col("claim_number")) == "", None)
        .otherwise(trim(col("claim_number")))
    )
    .withColumn(
        "denial_reason_std",
        initcap(trim(col("denial_reason")))
    )
    .drop("claim_number") 
    .drop("denial_reason")          
)

df_textcleaning.display()


In [0]:
# -----------------------------
# 7. AR Aging Bucket
# -----------------------------
from pyspark.sql.functions import col, when

df_AR_Bucket = df_textcleaning.withColumn(
    "ar_age_bucket",
    when(col("payment_lag_days").isNull(), "Unknown")

    .when(col("payment_lag_days") > 90, "90+")
    .when(col("payment_lag_days") > 60, "61-90")
    .when(col("payment_lag_days") > 30, "31-60")
    .when(col("payment_lag_days") > 0, "0-30")

    .otherwise("Collected")
)
df_AR_Bucket.display()

In [0]:
# -----------------------------
# 8. Period Column
# -----------------------------
from pyspark.sql.functions import date_format
df_period = df_AR_Bucket.withColumn(
        "bill_year_month",
        date_format(col("bill_dt"), "yyyy-MM")
    )

df_period.display()

In [0]:
# -----------------------------
# 9. Fully Paid Derived Flag
# -----------------------------
df_derived_flag = df_period.withColumn(
    "is_fully_paid",
    when(
        (col("outstanding_valid") == 0) &
        (trim(col("bill_status_std")) == "Paid"),
        1
    ).otherwise(0)
)


df_derived_flag.filter(col("bill_status_std")=="Paid").display()

In [0]:
# -----------------------------
# 10. Data Quality Flag
# -----------------------------
from pyspark.sql.functions import col, when

df_billing = df_derived_flag.withColumn(
    "dq_billing_flag",
    when(
        col("gross_amount_valid").isNull() |
        col("bill_dt").isNull() |
        (col("bill_status_std") == "Unknown"),
        1
    ).otherwise(0)
)

df_billing.display()

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "billing"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
df_billing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")